# Lesson 13: Decision Trees

*ML Academy — companion notebook*

Twenty questions for data: Gini impurity, entropy, information gain and pruning.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-13.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 11. In Python

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, stratify=data.target, random_state=0)

for depth in [1, 2, 3, 5, None]:          # None = grow until leaves are pure
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    print(f"max_depth={str(depth):4s} depth={tree.get_depth()} leaves={tree.get_n_leaves():2d} "
          f"train={tree.score(X_train, y_train):.3f} test={tree.score(X_test, y_test):.3f}")

tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
top = sorted(zip(tree.feature_importances_, data.feature_names), reverse=True)[:3]
print([(str(name), round(float(imp), 3)) for imp, name in top])

max_depth=1    depth=1 leaves= 2 train=0.930 test=0.888
max_depth=2    depth=2 leaves= 4 train=0.932 test=0.888
max_depth=3    depth=3 leaves= 8 train=0.977 test=0.916
max_depth=5    depth=5 leaves=18 train=1.000 test=0.902
max_depth=None depth=5 leaves=18 train=1.000 test=0.902
[('worst perimeter', 0.791), ('worst concave points', 0.115), ('worst radius', 0.04)]


## 14. Going deeper: pruning, honest importances and instability

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(Xtr, ytr)
print(" ccp_alpha   leaves   train acc   test acc")
for a in path.ccp_alphas[::3]:
    t = DecisionTreeClassifier(random_state=0, ccp_alpha=a).fit(Xtr, ytr)
    print(f"{a:9.4f}   {t.get_n_leaves():6d}   {t.score(Xtr, ytr):9.3f}   {t.score(Xte, yte):8.3f}")

 ccp_alpha   leaves   train acc   test acc
   0.0000       16       1.000      0.906
   0.0042       11       0.992      0.912
   0.0088        6       0.977      0.918
   0.0259        3       0.932      0.889


In [3]:
from sklearn.model_selection import GridSearchCV
search = GridSearchCV(DecisionTreeClassifier(random_state=0), {"ccp_alpha": path.ccp_alphas}, cv=5).fit(Xtr, ytr)
best = search.best_estimator_
print("alpha chosen by CV:", round(search.best_params_["ccp_alpha"], 4), " leaves:", best.get_n_leaves(), " test accuracy:", round(best.score(Xte, yte), 3))

alpha chosen by CV: 0.0088  leaves: 6  test accuracy: 0.918


In [4]:
from sklearn.inspection import permutation_importance

# Add a column of pure noise with many distinct values (like an ID number)
rng = np.random.default_rng(0)
Xn_tr = np.column_stack([Xtr, rng.permutation(len(ytr))])
Xn_te = np.column_stack([Xte, rng.permutation(len(yte))])
tree = DecisionTreeClassifier(random_state=0).fit(Xn_tr, ytr)
imp = tree.feature_importances_
perm = permutation_importance(tree, Xn_te, yte, n_repeats=30, random_state=0).importances_mean
print("noise column: impurity importance", round(imp[-1], 3), " rank", int((imp > imp[-1]).sum()) + 1, "of 31")
print("noise column: permutation importance on test data", round(perm[-1], 3))

noise column: impurity importance 0.011  rank 6 of 31
noise column: permutation importance on test data 0.0


In [5]:
# Instability: which feature does the root split use on bootstrap samples?
names = load_breast_cancer().feature_names
roots = {}
for _ in range(100):
    i = rng.integers(0, len(ytr), len(ytr))
    r = names[DecisionTreeClassifier(random_state=0).fit(Xtr[i], ytr[i]).tree_.feature[0]]
    roots[str(r)] = roots.get(str(r), 0) + 1
print(sorted(roots.items(), key=lambda kv: -kv[1]))

[('worst perimeter', 52), ('worst concave points', 30), ('worst radius', 8), ('mean concave points', 7), ('worst area', 3)]


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.tree import DecisionTreeClassifier, export_text
cols = ["size_m2", "dist_km", "age_yrs"]
tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(df[cols], df["sold_fast"])
print(export_text(tree, feature_names=cols))
print("training accuracy:", tree.score(df[cols], df["sold_fast"]))

|--- dist_km <= 11.50
|   |--- age_yrs <= 16.00
|   |   |--- class: 1
|   |--- age_yrs >  16.00
|   |   |--- class: 0
|--- dist_km >  11.50
|   |--- class: 0

training accuracy: 1.0


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [7]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Gini impurity

Compute the Gini impurity of a node from its class counts.

In [8]:
import numpy as np
counts = np.array([3, 1])
p = counts / counts.sum()
gini = None  # ← your code here
check(gini, 0.375, hint='1 − Σ p².')

❌ Not yet. Hint: 1 − Σ p².


### Challenge 2: Weighted impurity after a split

Combine the two children's Gini values, weighted by their sizes.

In [9]:
n_left, g_left = 5, 0.32
n_right, g_right = 5, 0.32
weighted = None  # ← your code here
check(weighted, 0.32, hint="Each child's Gini times its share of the examples.")

❌ Not yet. Hint: Each child's Gini times its share of the examples.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Gini impurity** — the missing line:

```python
gini = 1 - np.sum(p ** 2)
```

**2. Weighted impurity after a split** — the missing line:

```python
weighted = (n_left * g_left + n_right * g_right) / (n_left + n_right)
```